# Working with Text Files (pt. 2)

## Goals of this lecture

In the previous lecture, we discussed the basics of `open`ing a `.txt` file, as well as **reading** and **writing** to that file.

In this lecture, we'll talk about what you can do with a text file once you've already opened it. Because text files are `read` in as strings, you'll see many echoes of our previous lecture on [working with strings](strings).

- **Finding** a target `str` in a file.  
- **Counting** the number of words in a file.  
  - **Counting** how many times *each word* occurs.
  - Finding the most **frequent** word in a text.

- **JSON Files**:
    - What is a `.json` file? What is JSON more generally and why is it useful?  
    - How do we **read in** and **write** a `.json` file?
    - JSON files vs. JSON strings.

## Finding a target `str`

One common use case is **searching** a large volume of text to `return` particular sub-string.

- Where in the text does this sub-string occur?  
- What is the text surrounding one of its occurrences?

Note that this is not too far afield from a **search engine** like [Google](https://www.google.com/)!

### Our sample text

To start, we'll use a `.txt` file of [**Hamlet**](https://en.wikipedia.org/wiki/Hamlet), by [William Shakespeare](https://en.wikipedia.org/wiki/William_Shakespeare). The `.txt` file was retrieved from the [Project Gutenberg Corpus](https://www.gutenberg.org/browse/scores/top) online, and should be credited as such. 

First, let's use `readlines()` to extract each **line** of the play as a separate item in a list.

In [2]:
with open("data/hamlet.txt") as f:
    book = f.readlines()

#### Inspecting the text

In [ ]:
## This is just the title
book[0]

In [ ]:
## Partial list of characters in play
for line in book[5:12]:
    l = line.replace("\n", "")
    print(l)

#### Check-in

How could we check how many **lines** are in the `.txt` file?

In [3]:
### Your code here
len(book)

4934

### Finding a sample `str`

One of the most famous lines in *Hamlet* reads:

> To be, or not to be- that is the question...

Suppose we wanted to **find** the `str` `"that is the question"` in the book, and **return** the line number (at least in this `.txt` file).

How could we go about that?

### Solution: `enumerate`

- Use `enumerate` to iterate through each line of the play.  
- For each line, check if some `target_str` occurs in that line.  
- If it does, use `break` to **stop** iterating, and record which line it is.

In [ ]:
target_str = "that is the question"
for index, line in enumerate(book):
    if target_str in line:
        break
print("Line: {x}".format(x = line.replace("\n", "")))
print("Line number: {x}".format(x = index))

In [ ]:
## If we want to find any string, we can create a function that takes the target string as an input

def find_target(target_str):
    for index, line in enumerate(book):
        if target_str in line:
            break
    print("Line: {x}".format(x = line.replace("\n", "")))
    print("Line number: {x}".format(x = index))

find_target("that is the question")

### Check-in: Finding the next $N$ lines

What if we wanted to return the next $N$ (e.g., `5`) lines *after* this target string? 

- To do this, we just need to add another variable: `keep_lines`, which tells us *how many* additional lines we want to return.  
- Then, once we've retrieved the `index` of our `target_str`, we can **slice** between that `index` and `index + keep_lines`.

Try implementing this algorithm yourself first. 

**Hint**: The code can be *mostly* the same as before (i.e., use `enumerate`, etc.). 

In [ ]:
keep_lines = 5
target_str = "that is the question"

for index, line in enumerate(book):
    if target_str in line:
        print("Line: {x}".format(x = line.replace("\n", "")))
        print("Line number: {x}".format(x = index))
        
        # Now get and print the next keep_lines lines
        for i in range(index + 1, index + 1 + keep_lines):
            if i < len(book):  # Avoid index out of range
                print("Line: {x}".format(x = book[i].replace("\n", "")))
                print("Line number: {x}".format(x = i))
        break


### Check-in: What if `target_str` occurs multiple times?

What if we were looking for a more common `target_str`, e.g., one that occurred multiple times?  

1. What problems do you see with our previous approach (e.g., using `break` once we find `target_str`)?
2. How might you solve this problem? 

In [ ]:
target_str = "the question"
### Your answer here









#### Solution

In [ ]:
target_str = "the question"

for index, line in enumerate(book):
    if target_str in line:
        print("Line: {x}".format(x = line.replace("\n", "")))
        print("Line number: {x}".format(x = index))

### Check-in: Other considerations

These exercises really only scratch the surface of **searching** a file. Here are some other issues for consideration and discussion. 

How might you address:

1. Issues of **case**: e.g., what if *question* is spelled `"Question"`, not `"question"`?
2. Situations where a `target_str` spans multiple *lines*? 
3. Mismatch in punctuation, e.g., a misplaced `,`? 
4. A **partial match**, e.g., if $90\%$ of the characters match?

**Note**: These are challenging issues! And each of them likely has multiple solutions.

## Counting Words

Another very common **use case** is simply **counting** words.

- How many words are there overall?  
- How many *unique words* are used?  
- How many times does *each word* occur?  
- What is the *most frequent word*?

### Caveat: what *is* a word?

The question of what defines a word is surprisingly complex.

- First, languages have very different [**morphological systems**](https://en.wikipedia.org/wiki/Morphological_typology). So even *conceptually*, it's not always clear what makes a word "a word" in a given language.  
- Second, languages have very different [**writing systems**](https://en.wikipedia.org/wiki/Orthography). 
  - Some languages (like English, Spanish, etc.) have *spaces* between words in their written form.  
  - Other languages (like Classical Latin, Chinese, etc.) do [not typically use *spaces* between words](https://en.wikipedia.org/wiki/Scriptio_continua) in their written form.

Many **conceptual definitions** and **tools** for identifying *words* are rooted in English specifically, but those definitions and tools don't always generalize––languages can be very different.

### How *many* words?

The first question that might occur to us is *how many words* are in a book. 

To do this, we could:

- `read` the book in as one long `str`.  
- Use the `split` function to separate this long `str` by **spaces**, into a `list` of words.
- Count the number of items in this list.

#### Using `split`: a review

In [ ]:
sentence = "To be or not to be, that is the question"
sentence.split(" ")

#### Using `split` for Hamlet

In [ ]:
# First, read in as string
with open("data/hamlet.txt", "r") as f:
    book_str = f.read()

In [ ]:
# We should also clean up all those *newline* characters.
book_str = book_str.replace("\n", " ")
# To make it easier for later, we can also turn it into lowercase
book_str = book_str.lower()
# Now, use split to separate into words
book_words = book_str.split()
book_words[0:5]

In [ ]:
# How many items in list?
len(book_words)

### How many *unique* words?

Above, we calculated how many word *tokens* were in the book. 

- This means that the word "the" will be counted *every time* it occurs.  
- Instead, let's calculate the number of *unique* word types.

#### Using `set`

The `set` function will turn a `list` into a `set` object, which contains only the *unique elements* in that list.

In [ ]:
my_list = ["the", "dog", "is", "the", "best"]
set(my_list)

#### Check-in

Use the `set` function to calculate how many *unique* words are in this book.

In [ ]:
### Your code here

len(set(book_words))

### How many times does each word occur?

We might also want to know *how many times* each word occurs. 

- For example, perhaps "the" occurs $>1000$ times, whereas "question" occurs only ~$10$ times.  
- Ideally, we would store this in a `dict`:
   - Each **key** represents a *word*.  
   - Each **value** represents *how many times* that word occurred in *Hamlet*.

How might we go about this?

#### First pass: counting each word

As a first pass, let's use the following approach:

- First, create a `dict` to store our words.  
- Then, *iterate* through our `list` of words.  
- `if` a given word is not in our `dict`, add an entry for it (and set the value to `1`).  
- `if` a given word *is* in a `dict`, increase its value by `1`.

In [ ]:
word_counts = {}
for w in book_words:
    if w not in word_counts:
        word_counts[w] = 1
    else:
        word_counts[w] += 1

In [ ]:
# How many times does "the" occur?
word_counts['the']

In [ ]:
# How many times does "king" occur?
word_counts['king']

#### Check-in

Any issues with this **first pass** approach? 

**Hint**: One issue could have to do with punctuation...

Write a code that works when you may or may not have a '.' in sentence.

In [ ]:
### Your code here


#### Solution

One problem that you might've noticed is that words occurring at the *end of a sentence* don't have a space between the word and a period (e.g., `question.`). 

- This will *under-count* certain words.

To resolve this, we can `replace` all periods with an empty character before adding a word to our `dict`.

In [ ]:
word_counts = {}
for w in book_words:
    w_no_period = w.replace(".", "")
    if w_no_period not in word_counts:
        word_counts[w_no_period] = 1
    else:
        word_counts[w_no_period] += 1

In [ ]:
# How many times does "king" occur?
word_counts['king']

### Which word is most common?

Now that we have a `dict` representing how many times each word occurs, we can calculate **which word** is most common.

**Check-in**: Which word do you think is most frequent in *Hamlet*?

#### Finding the most frequent word

As always, there are multiple ways to do this.

But one simple approach is to:

- Use a `for` loop to iterate through all `items()` in the `dict`.  
- Track the `key_with_highest_value` we've seen so far.  
- Once the `for` loop is done, inspect `key_with_highest_value`.

In [ ]:
key_with_highest_value = None
max_count = 0
for word, count in word_counts.items():
    # If this word frequency > max_count
    if count > max_count:
        # Set new "highest word" to this word
        key_with_highest_value = word
        max_count = count


In [ ]:
## Now, inspect which word was most frequent
key_with_highest_value

#### Other approaches

There are *many different approaches* you could take to solving this problem. Some are more generalizable (but also more complicated) than what I've shown here.

- You can `sort` the dictionary by **value** (see the lecture on **dictionary operations**).  
- You could use the `max` function with `dict.get` as your `key` parameter (see below).

In [ ]:
# Another approach
max(word_counts, key = word_counts.get)

## JSON

## What is a `.json` file?

> A `.json` file is a **file written in the JSON file format**. It allows us to store structured data objects consisting of **key-value** pairs.

### What is JSON?

**JSON** = JavaScript Object Notation.

- Standard format for *representing* and *transmitting* data.  
   - "Standard" = different people/systems agree to use this format to send and receive information.  
- Represents data in **key-value** pairs.

### A Python `dict` is a collection of key-value pairs

A **dictionary** (`dict`) stores **key-value** pairs.

In [ ]:
my_class = {'Code': '1',
           'Department': 'CSS',
           'Instructor': 'Roman',
            'Prerequisite': True,
           'Enrollment': 120}
print(my_class)

In [ ]:
my_class['Department']

### JSON and `dict`: an analogy

Conceptually, JSON accomplishes the same goals as a Python `dict`.

- In fact, Python programmers often *convert* a `dict` into a JSON `str` when they want to store it in a file.  
- Similarly, you can **read in** a `.json` file and convert the contents into a `dict`.

**Bottom line**: we're not dealing with a fundamentally new data sturcture––it's another standardized way to represent **key-value pairs**.

## Reading in a `.json` file

Reading in a `.json` file shares some similarities with [reading `.txt` files](17-reading-text).  

- Must specify a **file path**.  
- File path can be either *absolute* or *relative*.

But there are also some important differences:

- To **read in** a `.json` file, we'll need to `import` the `json` library.  
- `json.load` will read in a **structured `.json` file** as a `dict`, not a `str`.

### Example: simple file

Here, we will work with a simple `.json` file: `data/restaurant.json`. 

- The file contains a structured representation of a restaurant.  
- We use `json.load(...)` to **load** this representation as a `dict`.

In [ ]:
## This imports the json library
import json

In [ ]:
## As with normal .txt. files, we use "open" to open the target restaurant
with open("data/restaurant.json", "r") as fp:
    ## use json.load to load as dict
    info = json.load(fp)

In [ ]:
info

### `load` creates a `dict`

Now, we can work with the **contents** of this file as we would any `dict`.

In [ ]:
info['Name']

In [ ]:
info['Location']

In [ ]:
info['Cuisine']

### Check-in

Try reading in another file that's stored in `data`: `data/school.json`. 

What is the value of the **Name** key?

In [ ]:
### Your code here

### Solution

In [ ]:
## As before, we use "open" to open the target file
with open("data/school.json", "r") as fp:
    ## use json.load to load as dict
    school_info = json.load(fp)

In [ ]:
## Get name of school
school_info['Name']

## Writing a `.json` file

Often, you'll want to **write** a structured `dict` to a file.  

- Useful for *storing* information, so you can access it later.  
- Useful for *transmitting* information between programs.  

We can use `json.dump(...)` to **write** (or "dump") a `dict` into a `.json` file.

### Simple example: course 

To start out, let's use the `my_class` dict we defined earlier.

In [ ]:
my_class['Code']

To **write** this to a file, we:

- `open` (create) a file with the name we want to call it.  
- Use `json.dump(dict_name, filename)`.

In [ ]:
with open("course.json", "w") as fp:
    json.dump(my_class, fp)

#### Checking that this worked

In [ ]:
with open("course.json", "r") as fp:
    course_info = json.load(fp)
print(course_info)

### Check-in

Create a new `dict` called `my_info`. Add the following keys/values:

- `Name`. 
- `Major`. 

Then, use `json.dump` to **write** this `dict` to a `.json` file called `my_info.json` to your own computer (in whichever directory you prefer).

In [ ]:
### Your code here
my_info = {
    'Name': 'Catarina',
    'Major': 'PoliSci'
}
with open('my_info.json', 'w') as f:
    json.dump(my_info, f)

## JSON files vs. JSON strings

The `load` and `dump` methods can be used to **read** and **write** a `dict` from/to a `.json` file.  

However, Python can also represent JSON as a **`str`**.

- To *read* a `dict` from a JSON `str`, use `loads` (load + *s*tring).  
- To *write* a `dict` into a JSON `str`, use `dumps` (dump + *s*tring).

### `json.dumps`

- Input: a `dict`. 
- Output: a JSON `str`.  

In [ ]:
json_str = json.dumps(my_class)
json_str

In [ ]:
type(json_str)

### `json.loads`

- Input: a JSON `str`.  
- Output: a `dict`.

### Other objects besides `dict`s

- Technically, you can use `dumps`/`loads` for other objects, such as `str`, `list`, and more.
- Though in my experience, a `dict` is the most common format.

In [ ]:
json.dumps([1, 2, 3])

In [ ]:
json.loads('[1, 2, 3]')

## Conclusion

This concludes our introduction to *working with text files*.

- The material in this lecture is a pre-cursor to basic Natural Language Processing techniques.  
- The material in the previous lecture covers the basics of **interacting with files** (`open`ing a file, using `read` and `write`, etc.). 
- This was also a super brief introduction to working with `.json` files.